# FIFA World Cup 2026 — Winner Prediction
## Notebook 02 — Feature Engineering

The goal of this notebook is to transform the raw datasets into a clean, 
model-ready feature matrix for training and simulation.

### Objectives
- Clean and prepare all 7 dataframes
- Engineer match-level features
- Merge datasets into a unified feature matrix
- Prepare the 2026 simulation inputs

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [48]:
df_matches = pd.read_csv("../data/processed/df_matches_clean.csv")
df_elo = pd.read_csv("../data/processed/df_elo_clean.csv")
df_train = pd.read_csv("../data/processed/df_train_clean.csv")
df_test = pd.read_csv("../data/processed/df_test_clean.csv")
df_matches_2026 = pd.read_csv("../data/processed/df_matches_2026_clean.csv")
df_teams_2026 = pd.read_csv("../data/processed/df_teams_2026_clean.csv")
df_stages_2026 = pd.read_csv("../data/processed/df_stages_2026_clean.csv")
df_host_cities = pd.read_csv("../data/processed/df_host_cities_clean.csv")

print("All dataframes loaded successfully.")

All dataframes loaded successfully.


In [49]:
# Drop irrelevant columns identified during EDA
cols_to_drop = [
    'Key Id', 'Match Id', 'Stadium Id', 'Home Team Id', 'Away Team Id',
    'Match Name', 'tournament Name', 'Match Time', 'Stadium Name',
    'City Name', 'Country Name', 'Home Team Code', 'Away Team Code',
    'Score', 'Replayed', 'Replay'
]

df_matches = df_matches.drop(columns=cols_to_drop)

In [50]:
# Parse Match Date from string to datetime
df_matches['Match Date'] = pd.to_datetime(df_matches['Match Date'])

In [51]:
# Recode legacy group stage formats to standard 'group stage'
# 'second group stage' was used in 1974 and 1982
# 'final round' was used in 1950
df_matches['Stage Name'] = df_matches['Stage Name'].replace({
    'second group stage': 'group stage',
    'final round': 'group stage'
})

df_matches['Stage Name'].value_counts()

Stage Name
group stage          718
round of 16           97
quarter-finals        70
semi-finals           38
final                 21
third-place match     20
Name: count, dtype: int64

In [52]:
# Encode Result as numeric target variable
# 0 = away team win, 1 = draw, 2 = home team win
df_matches['Result'] = df_matches['Result'].map({
    'away team win': 0,
    'draw': 1,
    'home team win': 2
})

df_matches['Result'].value_counts()

Result
2    545
0    240
1    179
Name: count, dtype: int64

In [53]:
# Filter to pre-tournament snapshot only
df_elo = df_elo[df_elo['snapshot_date'] == '2026-05-27']

# Drop irrelevant columns
df_elo = df_elo.drop(columns=['year', 'snapshot_date', 'country_code', 
                               'matches_home', 'matches_away', 'matches_neutral'])

df_elo.shape

(48, 17)

In [54]:
# Drop version column — just a dataset version tag, no predictive value
df_train = df_train.drop(columns=['version'])

# Drop squad_total_market_value_eur — 32 nulls (~17%), FIFA ranking already captures squad quality
df_train = df_train.drop(columns=['squad_total_market_value_eur'])

df_train.shape

(192, 22)

In [55]:
# Drop same columns as df_train for consistency
df_test = df_test.drop(columns=['version', 'squad_total_market_value_eur'])

df_test.shape

(48, 22)

In [56]:
# Drop irrelevant columns — venue name and airport code not needed for the model
df_host_cities = df_host_cities.drop(columns=['venue_name', 'airport_code'])

df_host_cities.head()

,id,city_name,country,region_cluster
0,1,Atlanta,USA,East
1,2,Boston,USA,East
2,3,Dallas,USA,Central
3,4,Houston,USA,Central
4,5,Kansas City,USA,Central


In [57]:
df_host_cities['region_cluster'].value_counts()

region_cluster
East       6
Central    6
West       4
Name: count, dtype: int64